In [9]:
pip install xgboost

Note: you may need to restart the kernel to use updated packages.


In [10]:
import pandas as pd
import numpy as np

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler

from xgboost import XGBRegressor
import joblib

In [11]:
df = pd.read_csv(r"C:\Users\ADMIN\Downloads\NIFTY50_all.csv")
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date')

In [12]:
df.head(-5)

,Date,Symbol,Series,Prev Close,Open,High,Low,Last,Close,VWAP,Volume,Turnover,Trades,Deliverable Volume,%Deliverble
75059,2000-01-03,HDFC,EQ,271.75,293.50,293.50,293.50,293.50,293.50,293.50,22744,6.675364e+11,NaN,NaN,NaN
117186,2000-01-03,IOC,EQ,254.00,260.00,273.25,250.00,267.35,270.85,258.55,23700,6.127648e+11,NaN,NaN,NaN
224580,2000-01-03,WIPRO,EQ,2522.40,2724.00,2724.20,2724.00,2724.20,2724.20,2724.17,1599,4.355942e+11,NaN,NaN,NaN
187156,2000-01-03,TELCO,EQ,201.60,207.40,217.25,207.40,217.00,216.75,214.28,676126,1.448775e+13,NaN,NaN,NaN
205542,2000-01-03,TITAN,EQ,144.95,146.00,156.45,146.00,155.00,155.70,154.36,23000,3.550370e+11,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46262,2021-04-30,CIPLA,EQ,906.50,900.75,921.00,900.75,910.00,910.35,911.47,6459737,5.887824e+14,121466.0,2004555.0,0.3103
176864,2021-04-30,SBIN,EQ,359.40,353.45,362.50,350.45,352.30,353.50,357.11,53832840,1.922408e+15,296959.0,13261927.0,0.2464
205541,2021-04-30,TECHM,EQ,976.90,973.80,976.90,955.40,958.85,960.40,968.21,3129105,3.029643e+14,70097.0,2104528.0,0.6726
131791,2021-04-30,JSWSTEEL,EQ,726.50,719.60,740.00,711.45,713.70,717.85,723.92,36449711,2.638675e+15,376696.0,3416355.0,0.0937


In [13]:
df.sample(5)

,Date,Symbol,Series,Prev Close,Open,High,Low,Last,Close,VWAP,Volume,Turnover,Trades,Deliverable Volume,%Deliverble
20311,2021-03-22,BAJAJFINSV,EQ,9440.65,9450.0,9494.95,9312.05,9402.00,9405.85,9409.56,293029,2.757274e+14,39865.0,55008.0,0.1877
143735,2011-02-09,M&M,EQ,627.70,629.5,666.70,618.90,659.00,655.20,650.91,6402308,4.167339e+14,NaN,3151394.0,0.4922
51031,2008-08-27,DRREDDY,EQ,573.70,579.0,597.05,575.00,590.00,589.70,589.51,470904,2.776031e+13,NaN,257053.0,0.5459
186377,2018-03-07,SUNPHARMA,EQ,531.75,532.0,534.15,520.10,523.35,524.85,526.19,6282066,3.305558e+14,67981.0,1509301.0,0.2403
80254,2020-11-17,HDFC,EQ,2311.25,2339.9,2364.75,2307.40,2344.85,2349.00,2332.31,7393052,1.724292e+15,173799.0,4613099.0,0.6240


In [14]:
df['MA_5'] = df['Close'].rolling(5).mean().shift(1) #This is the step where you are getting wrong values
df['MA_10'] = df['Close'].rolling(10).mean().shift(1)

# add Lag features
df['lag_1'] = df['Close'].shift(1)
df['lag_2'] = df['Close'].shift(2)
df['lag_3'] = df['Close'].shift(3)


df['volatility'] = df['Close'].rolling(5).std().shift(1)

df.dropna(inplace=True)

In [15]:
features = ['MA_5', 'MA_10', 'lag_1', 'lag_2', 'lag_3', 'volatility', 'Volume']
X = df[features]
y = df['Close']

In [16]:
tscv = TimeSeriesSplit(n_splits=5)

In [17]:
model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    random_state=42
)

In [18]:
for train_idx, test_idx in tscv.split(X):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)

print("Final RMSE:", rmse)
print("Final MAE:", mae)

Final RMSE: 3547.8297787713277
Final MAE: 1527.8965906522337


In [19]:
from sklearn.metrics import r2_score
r2 = r2_score(y_test, preds)
print("R2 Score:", r2)

R2 Score: 0.346627234909177


In [20]:
joblib.dump(model, 'xgb_nifty_model.pkl')

['xgb_nifty_model.pkl']